# Vergleich der Translations-Modelle (SFT vs. DPO vs. Baseline)
### Masterarbeit: Automatische Übersetzung von Alltagssprache (AS) in Leichte Sprache (LS)

Dieses Notebook vergleicht die Übersetzungen von:
1. **Baseline mBART-50 (untrainiert)**
2. **SFT Modell (Supervised Fine-Tuned)**
3. **DPO Modell (Direct Preference Optimized)**

Wir evaluieren die Übersetzungen qualitativ und quantitativ über den Simplicity Score der Regressoren.


In [ ]:
import os
import json
import torch
import torch.nn as nn
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
import spacy
import pandas as pd

while not os.path.exists(".git"):
    parent = os.path.dirname(os.getcwd())
    if parent == os.getcwd(): break
    os.chdir("..")
print("Arbeitsverzeichnis:", os.getcwd())


In [ ]:
MODEL_NAME = "facebook/mbart-large-50"
DEVICE = torch.device("cuda" if torch.cuda.is_available() else ("mps" if torch.backends.mps.is_available() else "cpu"))
print("Nutze Device:", DEVICE)


## 1. Alle Modelle und Tokenizer laden


In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
tokenizer.src_lang = "de_DE"
tokenizer.tgt_lang = "de_DE"

# Baseline
model_base = AutoModelForSeq2SeqLM.from_pretrained(MODEL_NAME).to(DEVICE)
model_base.eval()

# SFT
model_sft = AutoModelForSeq2SeqLM.from_pretrained(MODEL_NAME).to(DEVICE)
if os.path.exists("results/models/2_sft.pt"):
    model_sft.load_state_dict(torch.load("results/models/2_sft.pt", map_location=DEVICE))
    print("SFT geladen.")
model_sft.eval()

# DPO
import glob
model_dpo = AutoModelForSeq2SeqLM.from_pretrained(MODEL_NAME).to(DEVICE)
dpo_loaded = False
dpo_paths = glob.glob("results/models/seq2seq_dpo*")
for path in dpo_paths:
    try:
        if os.path.isdir(path):
            model_dpo = AutoModelForSeq2SeqLM.from_pretrained(path).to(DEVICE)
        else:
            model_dpo.load_state_dict(torch.load(path, map_location=DEVICE))
        print(f"DPO von {path} geladen.")
        dpo_loaded = True
        break
    except:
        continue
model_dpo.eval()
print("Modell-Laden abgeschlossen!")


## 2. Simplicity-Regressor zur Bewertung laden


In [ ]:
class BiLSTMRegressor(nn.Module):
    def __init__(self, vocab_size, embed_dim=128, hidden_dim=128, dropout=0.3):
        super(BiLSTMRegressor, self).__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=0)
        self.lstm = nn.LSTM(embed_dim, hidden_dim, batch_first=True, bidirectional=True)
        self.fc = nn.Linear(hidden_dim * 2, 1)
        self.dropout = nn.Dropout(dropout)
        self.sigmoid = nn.Sigmoid()
    def forward(self, x):
        embedded = self.dropout(self.embedding(x))
        _, (hidden, _) = self.lstm(embedded)
        hidden = torch.cat((hidden[-2,:,:], hidden[-1,:,:]), dim=1)
        out = self.fc(self.dropout(hidden))
        return self.sigmoid(out)

nlp = spacy.load("de_core_news_sm", disable=["ner", "tagger", "lemmatizer"])
reg_model = None
reg_stoi = None

if os.path.exists("data/vocabs/synthetic_vocab.json") and os.path.exists("results/models/bilstm_synthetic_regression.pt"):
    with open("data/vocabs/synthetic_vocab.json", "r", encoding="utf-8") as f:
        reg_stoi = json.load(f)
    reg_model = BiLSTMRegressor(len(reg_stoi)).to(DEVICE)
    reg_model.load_state_dict(torch.load("results/models/bilstm_synthetic_regression.pt", map_location=DEVICE))
    reg_model.eval()
    print("Simplicity Regressor geladen!")


## 3. Übersetzungs- und Bewertungspipeline


In [ ]:
def get_score(text):
    if reg_model is None or reg_stoi is None: return 0.0
    tokens = [t.text.lower() for t in nlp(text) if not t.is_space]
    encoded = [reg_stoi.get(t, reg_stoi.get("<unk>", 1)) for t in tokens]
    inp = torch.tensor([encoded], dtype=torch.long).to(DEVICE)
    with torch.no_grad():
        return reg_model(inp).squeeze().item()

def translate_one(model, text):
    src_text = text
    inputs = tokenizer(src_text, return_tensors="pt", max_length=256, truncation=True).to(DEVICE)
    with torch.no_grad():
        gen = model.generate(
            input_ids=inputs["input_ids"],
            attention_mask=inputs["attention_mask"],
            max_length=256,
            num_beams=4,
            early_stopping=True,
            forced_bos_token_id=tokenizer.lang_code_to_id["de_DE"] if hasattr(tokenizer, "lang_code_to_id") else None
        )
    return tokenizer.decode(gen[0], skip_special_tokens=True)

def compare_models(text):
    trans_base = translate_one(model_base, text)
    trans_sft = translate_one(model_sft, text)
    trans_dpo = translate_one(model_dpo, text)
    
    print(f"AS: {text} [Simplicity: {get_score(text):.4f}]")
    print(f"-> Baseline: {trans_base} [Simplicity: {get_score(trans_base):.4f}]")
    print(f"-> SFT:      {trans_sft} [Simplicity: {get_score(trans_sft):.4f}]")
    print(f"-> DPO:      {trans_dpo} [Simplicity: {get_score(trans_dpo):.4f}]")
    print("=" * 60)

compare_models("Wenn Sie krank sind, müssen Sie eine Arbeitsunfähigkeitsbescheinigung beim Arbeitgeber einreichen.")
compare_models("Die gesetzliche Rentenversicherung sichert Arbeitnehmer im Alter finanziell ab.")


## 4. Eigene Sätze testen


In [ ]:
custom_sentence = "Schreiben Sie hier Ihren eigenen Satz zum Vergleichen..."
compare_models(custom_sentence)
